In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from pillow_lab_rotation.simulate import CTDSSim
from pillow_lab_rotation.tools import vec
from scipy.linalg import block_diag
from sklearn.decomposition import NMF

# To initialize the CTDS model we first fit a linear dynamical system (linear RNN) by solving a constrained regression problem
$$
\boldsymbol{y}_{t+1} = \boldsymbol{Jy}_t + \boldsymbol{v}_t \qquad \text{s.t. }\boldsymbol{J}\text{ obeys Dale's law}
$$

$$
||y_{t+1} - Jy_t||_2^2 = (y_{t+1} - Jy_t)^\top(y_{t+1} - Jy_t)
$$

We can rewrite this as a quadratic programming problem. Let $Y$ be the matrix of $t + 1$ timepoints and $X$ be the matrix of $t$ timepoints. We can rewrite the problem as so

$$
\min -2\text{vec}(Y^\top X)^\top\text{vec}(J) + \text{vec}(J)^\top(I \otimes X^\top X)\text{vec}(J)
$$
Such that $J$ obey's Dale's law

In [8]:
Ne, Ni = 5, 5
N = Ne + Ni
De, Di = 2, 3
D = De + Di
T = 50

simulation = CTDSSim(De, Di, Ne, Ni)
np.random.seed(42)
simulation.create_params()
_, observations = simulation.simulate(T, 100)

Y = observations[:, 1:]
X = observations[:, :-1]
Y = Y.reshape(-1, 10)
X = X.reshape(-1, 10)

(100, 49, 10, 1)


In [ ]:
class EIRNNInit:
    def __init__(
            self,
            Ne: int,
            Ni: int,
            De: int,
            Di: int
    ):
        self.Ne = Ne
        self.Ni = Ni
        self.N = Ne + Ni
        self.De = De
        self.Di = Di
        self.r = De + Di

    def fit(
            self,
            observations: np.ndarray
    ):
        '''
        observations are expected to be shape (trials, time, N, 1)
        '''
        
        self.Y_t = observations[:, :-1]
        self.Y_tp1 = observations[:, 1:]
        self.Y_t = self.Y_t.reshape(-1, self.N)
        self.Y_tp1 = self.Y_tp1.reshape(-1, self.N)

    
    def _solve_constrained_regression(self):
        J = cp.Variable(shape=(self.N, self.N))

        constraints = [
            J[:, :self.Ne] >= 0,
            J[:, self.Ne:] <= 0
        ]

        objective = cp.Minimize(
            cp.norm(self.Y_tp1.T - J @ self.Y_t.T, 'fro')
        )
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.MOSEK, verbose=False)
        self.J_fit = J.value

    def _nmf(self, n_runs=10, seed=0):
        Je = np.abs(self.J_fit[:self.Ne])
        Ji = np.abs(self.J_fit[self.Ne:])

        best_error = np.inf
        for i in range(n_runs):
            e_model = NMF(n_components=self.De, init='random', random_state=seed + i)
            W_e = e_model.fit_transform(Je)
            H_e = e_model.components_

            i_model = NMF(n_components=self.Di, init='random', random_state=seed + i)
            W_i = i_model.fit_transform(Ji)
            H_i = i_model.components_

            U = block_diag(W_e, W_i)
            V = np.vstack([H_e, H_i])
            V[:, self.Ne:] *= -1

            error = np.linalg.norm(U @ V - self.J_fit, 'fro') / np.linalg.norm(self.J_fit, 'fro')
            if error < best_error:
                best_error = error
                self.U, self.V = U, V


#